In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm
import matplotlib
import lightgbm as lgb
from xgboost import XGBClassifier
from sklearn.model_selection import cross_validate
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder

import re

font_path = "C:/Windows/Fonts/gulim.ttc"
font = fm.FontProperties(fname=font_path).get_name()
matplotlib.rc("font", family=font)

In [2]:
df_team = pd.read_csv(
    "../../EDA/Merge/data/result data/Final DF.csv"
)  # Year, Nation, ..., DEF_INDEX 등

# 예시: 경기 단위 데이터
df_match = pd.read_csv("matches_1930_2022.csv")

In [3]:
df_team.columns
df_team = df_team[
    [
        "Year",
        "Nation",
        "Eng_Nation",
        "Q_WR",
        "Q_GR",
        "F_Rank",
        "F_Point",
        "F_Rd",
        "F_Pd",
        "Avg_Apps",
        "Avg_Age",
        "Avg_Famous",
        "FS_0",
        "FS_1",
        "FS_2",
        "FS_3",
        "FS_4",
        "FS_5",
        "FS_6",
        "FS_7",
        "FS_8",
        "FS_9",
        "FS_10",
        "FS_11",
        "FS_12",
        "FS_13",
    ]
]

In [4]:
df_match = df_match[
    ["Year", "home_team", "away_team", "home_score", "away_score", "Score"]
]
df_match

,Year,home_team,away_team,home_score,away_score,Score
0,2022,Argentina,France,3,3,(4) 3–3 (2)
1,2022,Croatia,Morocco,2,1,2–1
2,2022,France,Morocco,2,0,2–0
3,2022,Argentina,Croatia,3,0,3–0
4,2022,Morocco,Portugal,1,0,1–0
...,...,...,...,...,...,...
959,1930,Argentina,France,1,0,1–0
960,1930,Yugoslavia,Brazil,2,1,2–1
961,1930,Romania,Peru,3,1,3–1
962,1930,United States,Belgium,3,0,3–0


In [5]:
country_map = {
    "브라질": "Brazil",
    "독일": "Germany",
    "터키": "Türkiye",
    "대한민국": "Korea Republic",
    "스페인": "Spain",
    "잉글랜드": "England",
    "세네갈": "Senegal",
    "미국": "United States",
    "일본": "Japan",
    "덴마크": "Denmark",
    "멕시코": "Mexico",
    "아일랜드": "Republic of Ireland",
    "스웨덴": "Sweden",
    "벨기에": "Belgium",
    "이탈리아": "Italy",
    "파라과이": "Paraguay",
    "남아프리카 공화국": "South Africa",
    "아르헨티나": "Argentina",
    "코스타리카": "Costa Rica",
    "카메룬": "Cameroon",
    "포르투갈": "Portugal",
    "러시아": "Russia",
    "크로아티아": "Croatia",
    "에콰도르": "Ecuador",
    "폴란드": "Poland",
    "우루과이": "Uruguay",
    "나이지리아": "Nigeria",
    "프랑스": "France",
    "튀니지": "Tunisia",
    "슬로베니아": "Slovenia",
    "중국": "China PR",
    "사우디아라비아": "Saudi Arabia",
    "우크라이나": "Ukraine",
    "스위스": "Switzerland",
    "네덜란드": "Netherlands",
    "가나": "Ghana",
    "호주": "Australia",
    "코트디부아르": "Côte d'Ivoire",
    "체코": "Czech Republic",
    "앙골라": "Angola",
    "이란": "IR Iran",
    "트리니다드 토바고": "Trinidad and Tobago",
    "토고": "Togo",
    "세르비아 몬테네그로": "Serbia and Montenegro",
    "칠레": "Chile",
    "슬로바키아": "Slovakia",
    "뉴질랜드": "New Zealand",
    "세르비아": "Serbia",
    "그리스": "Greece",
    "알제리": "Algeria",
    "온두라스": "Honduras",
    "북한": "Korea DPR",
    "콜롬비아": "Colombia",
    "보스니아 헤르체고비나": "Bosnia and Herzegovina",
    "페루": "Peru",
    "모로코": "Morocco",
    "아이슬란드": "Iceland",
    "이집트": "Egypt",
    "파나마": "Panama",
    "웨일스": "Wales",
    "캐나다": "Canada",
    "카타르": "Qatar",
}

In [6]:
# ---------------------------
# 0️⃣ 2002년 이후 경기만 선택
# ---------------------------
df_match_filtered = df_match[df_match["Year"] >= 2002].reset_index(drop=True)

# ---------------------------
# 1️⃣ 팀 데이터에 영어 이름 컬럼 추가 (country_map 사용)
# ---------------------------
df_team["Eng_Nation"] = df_team["Nation"].map(country_map)

# ---------------------------
# 2️⃣ Year + Eng_Nation 기준 인덱스 설정
# ---------------------------
df_team_renamed = df_team.set_index(["Year", "Eng_Nation"])

# ---------------------------
# 3️⃣ 매핑 실패 팀 기록용
# ---------------------------
missing_teams = set()


# ---------------------------
# 4️⃣ 점수 문자열에서 최종 결과 계산
# ---------------------------
def parse_result_from_score(score_str):
    """
    score_str 예시: "3–3", "(4) 3–3 (2)", "2–1"
    반환: 1 = 홈승, 0 = 무승부, -1 = 홈패
    """
    # 정규/연장 점수 추출
    main_match = re.search(r"(\d+)\s*–\s*(\d+)", score_str)
    if not main_match:
        return None  # 점수 형식 이상

    home_main = int(main_match.group(1))
    away_main = int(main_match.group(2))

    # 점수가 다르면 정규/연장으로 승패 결정
    if home_main > away_main:
        return 1
    elif home_main < away_main:
        return -1
    else:
        # 동점이면 승부차기 점수 확인
        pens = re.findall(r"\((\d+)\)", score_str)
        if len(pens) == 2:
            home_pen = int(pens[0])
            away_pen = int(pens[1])
            if home_pen > away_pen:
                return 1
            elif home_pen < away_pen:
                return -1
        # 승부차기 없거나 동점이면 무승부
        return 0


# ---------------------------
# 5️⃣ 홈/어웨이 팀 특징 매핑 함수
# ---------------------------
def map_team_features_safe(row):
    year = row["Year"]
    home = row["home_team"]
    away = row["away_team"]

    # 홈팀 특징
    try:
        home_feat = df_team_renamed.loc[(year, home)].add_prefix("Home_")
    except KeyError:
        missing_teams.add(home)
        home_feat = pd.Series(dtype=float)

    # 어웨이팀 특징
    try:
        away_feat = df_team_renamed.loc[(year, away)].add_prefix("Away_")
    except KeyError:
        missing_teams.add(away)
        away_feat = pd.Series(dtype=float)

    # 경기 결과 계산 (홈승=1, 무승부=0, 홈패=-1)
    result = parse_result_from_score(row["Score"])

    return pd.concat([home_feat, away_feat, pd.Series({"Result": result})])


# ---------------------------
# 6️⃣ 적용
# ---------------------------
df_final = df_match_filtered.apply(map_team_features_safe, axis=1).reset_index(
    drop=True
)

# ---------------------------
# 7️⃣ 매핑 실패 팀 확인
# ---------------------------
if missing_teams:
    print("매핑이 안 된 팀:", missing_teams)

# ---------------------------
# 8️⃣ 결과 확인
# ---------------------------
print(df_final.head())

  Home_Nation  Home_Q_WR  Home_Q_GR  Home_F_Rank  Home_F_Point  Home_F_Rd  \
0       아르헨티나       0.52   3.888889         8.33       1654.10       6.58   
1       크로아티아       0.54   1.479452        11.00       1620.42      -4.00   
2         프랑스       0.63   2.830189         2.67       1741.10     -10.66   
3       아르헨티나       0.52   3.888889         8.33       1654.10       6.58   
4         모로코       0.51   3.055556        38.33       1470.74     -28.00   

   Home_F_Pd  Home_Avg_Apps  Home_Avg_Age  Home_Avg_Famous  ...  Away_FS_5  \
0     190.35        0.50000         27.81                9  ...      56.58   
1     592.92        0.46154         27.42                3  ...      54.26   
2     644.77        0.65385         26.69                7  ...      54.26   
3     190.35        0.50000         27.81                9  ...      59.85   
4     938.74        0.23077         26.31                2  ...      59.48   

   Away_FS_6  Away_FS_7  Away_FS_8  Away_FS_9  Away_FS_10  Away_FS_1

In [7]:
df_final

,Home_Nation,Home_Q_WR,Home_Q_GR,Home_F_Rank,Home_F_Point,Home_F_Rd,Home_F_Pd,Home_Avg_Apps,Home_Avg_Age,Home_Avg_Famous,...,Away_FS_5,Away_FS_6,Away_FS_7,Away_FS_8,Away_FS_9,Away_FS_10,Away_FS_11,Away_FS_12,Away_FS_13,Result
0,아르헨티나,0.52,3.888889,8.33,1654.10,6.58,190.35,0.50000,27.81,9,...,56.58,71.88,68.46,74.35,68.04,73.38,17.92,70.40,57.00,1
1,크로아티아,0.54,1.479452,11.00,1620.42,-4.00,592.92,0.46154,27.42,3,...,54.26,66.91,63.43,64.09,65.09,67.65,20.72,60.93,45.98,1
2,프랑스,0.63,2.830189,2.67,1741.10,-10.66,644.77,0.65385,26.69,7,...,54.26,66.91,63.43,64.09,65.09,67.65,20.72,60.93,45.98,1
3,아르헨티나,0.52,3.888889,8.33,1654.10,6.58,190.35,0.50000,27.81,9,...,59.85,69.46,68.35,69.38,66.50,68.27,15.12,64.31,53.09,1
4,모로코,0.51,3.055556,38.33,1470.74,-28.00,938.74,0.23077,26.31,2,...,59.48,68.76,72.64,67.80,71.44,68.72,17.90,66.90,52.27,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
379,스페인,0.65,3.227273,5.00,742.33,0.00,0.00,0.26087,27.13,3,...,57.71,60.50,65.93,66.57,60.14,70.64,28.54,55.18,62.57,1
380,아일랜드,0.56,2.225000,31.33,598.00,0.00,0.00,0.21739,27.52,2,...,54.33,59.62,70.86,72.10,68.48,71.90,30.24,58.98,60.50,0
381,우루과이,0.43,1.094737,31.33,597.67,0.00,0.00,0.00000,25.83,2,...,59.00,63.24,72.95,77.57,77.24,83.76,31.33,67.98,62.26,-1
382,독일,0.55,1.817073,8.67,718.33,0.00,0.00,0.21739,28.26,10,...,61.00,57.10,68.00,66.38,68.05,75.10,26.64,55.93,62.43,1


In [8]:
df_final.columns

Index(['Home_Nation', 'Home_Q_WR', 'Home_Q_GR', 'Home_F_Rank', 'Home_F_Point',
       'Home_F_Rd', 'Home_F_Pd', 'Home_Avg_Apps', 'Home_Avg_Age',
       'Home_Avg_Famous', 'Home_FS_0', 'Home_FS_1', 'Home_FS_2', 'Home_FS_3',
       'Home_FS_4', 'Home_FS_5', 'Home_FS_6', 'Home_FS_7', 'Home_FS_8',
       'Home_FS_9', 'Home_FS_10', 'Home_FS_11', 'Home_FS_12', 'Home_FS_13',
       'Away_Nation', 'Away_Q_WR', 'Away_Q_GR', 'Away_F_Rank', 'Away_F_Point',
       'Away_F_Rd', 'Away_F_Pd', 'Away_Avg_Apps', 'Away_Avg_Age',
       'Away_Avg_Famous', 'Away_FS_0', 'Away_FS_1', 'Away_FS_2', 'Away_FS_3',
       'Away_FS_4', 'Away_FS_5', 'Away_FS_6', 'Away_FS_7', 'Away_FS_8',
       'Away_FS_9', 'Away_FS_10', 'Away_FS_11', 'Away_FS_12', 'Away_FS_13',
       'Result'],
      dtype='object')

In [9]:
df = df_final

In [10]:
# 홈/어웨이 컬럼 분리
home_cols = [col for col in df.columns if col.startswith("Home_")]
away_cols = [col for col in df.columns if col.startswith("Away_")]

# 컬럼 이름 접두사 바꾸기
home_rename = {col: col.replace("Home_", "Away_") for col in home_cols}
away_rename = {col: col.replace("Away_", "Home_") for col in away_cols}

# 데이터 복사 후 컬럼 교체
df_swapped = df.copy()
df_swapped = df_swapped.rename(columns={**home_rename, **away_rename})

# Result 반대로 바꾸기
df_swapped["Result"] = df_swapped["Result"].replace({1: -1, -1: 1, 0: 0})

# 원본과 합치기
df_expanded = pd.concat([df, df_swapped], ignore_index=True)

print(df_expanded.shape)
print(df_expanded.head())

(768, 49)
  Home_Nation  Home_Q_WR  Home_Q_GR  Home_F_Rank  Home_F_Point  Home_F_Rd  \
0       아르헨티나       0.52   3.888889         8.33       1654.10       6.58   
1       크로아티아       0.54   1.479452        11.00       1620.42      -4.00   
2         프랑스       0.63   2.830189         2.67       1741.10     -10.66   
3       아르헨티나       0.52   3.888889         8.33       1654.10       6.58   
4         모로코       0.51   3.055556        38.33       1470.74     -28.00   

   Home_F_Pd  Home_Avg_Apps  Home_Avg_Age  Home_Avg_Famous  ...  Away_FS_5  \
0     190.35        0.50000         27.81                9  ...      56.58   
1     592.92        0.46154         27.42                3  ...      54.26   
2     644.77        0.65385         26.69                7  ...      54.26   
3     190.35        0.50000         27.81                9  ...      59.85   
4     938.74        0.23077         26.31                2  ...      59.48   

   Away_FS_6  Away_FS_7  Away_FS_8  Away_FS_9  Away_FS_10 

In [11]:
home_cols = [col for col in df_final.columns if col.startswith("Home_")]
away_cols = [col for col in df_final.columns if col.startswith("Away_")]

# Nation 같은 문자열 컬럼 제외
home_cols = [c for c in home_cols if c not in ["Home_Nation"]]
away_cols = [c for c in away_cols if c not in ["Away_Nation"]]

# 공통 지표 이름 추출 (Home_ 접두사 제거)
metrics = [col.replace("Home_", "") for col in home_cols]

diff_data = {}

for metric in metrics:
    home_col = f"Home_{metric}"
    away_col = f"Away_{metric}"

    # 두 컬럼이 존재하고 수치형일 때만 처리
    if home_col in df_final.columns and away_col in df_final.columns:
        if pd.api.types.is_numeric_dtype(
            df_final[home_col]
        ) and pd.api.types.is_numeric_dtype(df_final[away_col]):
            diff_data[metric] = df_final[home_col] - df_final[away_col]

# 결과 컬럼 추가
diff_data["Result"] = df_final["Result"]

# 최종 데이터프레임 생성
df_diff = pd.DataFrame(diff_data)

print(df_diff.shape)
print(df_diff.head())

(384, 24)
   Q_WR      Q_GR  F_Rank  F_Point   F_Rd    F_Pd  Avg_Apps  Avg_Age  \
0 -0.11  1.058700    5.66   -87.00  17.24 -454.42  -0.15385     1.12   
1  0.03 -1.576104  -27.33   149.68  24.00 -345.82   0.23077     1.11   
2  0.12 -0.225367  -35.66   270.36  17.34 -293.97   0.42308     0.38   
3 -0.02  2.409437   -2.67    33.68  10.58 -402.57   0.03846     0.39   
4 -0.16 -0.476359   32.66  -186.23 -28.17  528.27  -0.38461    -0.57   

   Avg_Famous  FS_0  ...  FS_5  FS_6  FS_7   FS_8  FS_9  FS_10  FS_11  FS_12  \
0           2 -1.34  ...  5.92  1.93  2.08  -5.54  1.69  -1.96  -0.25  -1.94   
1           1  0.76  ...  5.59  2.55  4.92   5.29  1.41   0.62  -5.60   3.38   
2           5  7.76  ...  2.32  4.97  5.03  10.26  2.95   5.73  -2.80   9.47   
3           6  5.66  ...  2.65  4.35  2.19  -0.57  3.23   3.15   2.55   4.15   
4          -9 -5.61  ... -5.22 -1.85 -9.21  -3.71 -6.35  -1.07   2.82  -5.97   

   FS_13  Result  
0  -0.90       1  
1   7.11       1  
2  11.02       1  


In [12]:
df_diff.columns

Index(['Q_WR', 'Q_GR', 'F_Rank', 'F_Point', 'F_Rd', 'F_Pd', 'Avg_Apps',
       'Avg_Age', 'Avg_Famous', 'FS_0', 'FS_1', 'FS_2', 'FS_3', 'FS_4', 'FS_5',
       'FS_6', 'FS_7', 'FS_8', 'FS_9', 'FS_10', 'FS_11', 'FS_12', 'FS_13',
       'Result'],
      dtype='object')

In [13]:
model = lgb.LGBMClassifier(
    n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42, verbose=-1
)

cross_val_score(
    model,
    df_expanded.drop(["Home_Nation", "Away_Nation", "Result"], axis=1),
    df_expanded["Result"],
    cv=5,
)

array([0.49350649, 0.48701299, 0.41558442, 0.50326797, 0.47712418])

In [14]:
X = df_expanded.drop(["Home_Nation", "Away_Nation", "Result"], axis=1)

In [15]:
poly = PolynomialFeatures(degree=3, include_bias=False)
X_poly = poly.fit_transform(X)

In [16]:
pca = PCA(n_components=16)
X_p = pca.fit_transform(X_poly)
model = lgb.LGBMClassifier(
    n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42, verbose=-1
)

cross_val_score(model, X_p, df_expanded["Result"], cv=5).mean()

c:\Users\user\anaconda3\envs\ml_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\user\anaconda3\envs\ml_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\user\anaconda3\envs\ml_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\user\anaconda3\envs\ml_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\user\anaconda3\envs\ml_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature nam

np.float64(0.522239198709787)

In [17]:
df_diff

,Q_WR,Q_GR,F_Rank,F_Point,F_Rd,F_Pd,Avg_Apps,Avg_Age,Avg_Famous,FS_0,...,FS_5,FS_6,FS_7,FS_8,FS_9,FS_10,FS_11,FS_12,FS_13,Result
0,-0.11,1.058700,5.66,-87.00,17.24,-454.42,-0.15385,1.12,2,-1.34,...,5.92,1.93,2.08,-5.54,1.69,-1.96,-0.25,-1.94,-0.90,1
1,0.03,-1.576104,-27.33,149.68,24.00,-345.82,0.23077,1.11,1,0.76,...,5.59,2.55,4.92,5.29,1.41,0.62,-5.60,3.38,7.11,1
2,0.12,-0.225367,-35.66,270.36,17.34,-293.97,0.42308,0.38,5,7.76,...,2.32,4.97,5.03,10.26,2.95,5.73,-2.80,9.47,11.02,1
3,-0.02,2.409437,-2.67,33.68,10.58,-402.57,0.03846,0.39,6,5.66,...,2.65,4.35,2.19,-0.57,3.23,3.15,2.55,4.15,3.01,1
4,-0.16,-0.476359,32.66,-186.23,-28.17,528.27,-0.38461,-0.57,-9,-5.61,...,-5.22,-1.85,-9.21,-3.71,-6.35,-1.07,2.82,-5.97,-6.29,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
379,0.19,2.044174,-35.00,171.66,0.00,0.00,0.26087,-1.57,2,15.16,...,11.29,9.27,9.57,11.66,17.95,12.63,6.51,14.62,1.98,1
380,0.16,0.344048,-11.00,30.33,0.00,0.00,-0.43478,2.13,1,3.03,...,-0.18,-5.57,2.59,2.35,6.27,10.95,4.09,0.70,4.50,0
381,-0.09,-0.556057,14.00,-61.33,0.00,0.00,-0.26087,-2.26,0,-3.81,...,-5.05,-8.38,-8.04,-6.34,-9.42,-6.21,-0.31,-11.34,-2.99,-1
382,-0.02,-0.949593,-37.00,156.00,0.00,0.00,-0.17391,1.83,10,12.19,...,-7.00,6.19,-1.62,1.48,4.52,5.47,10.96,6.95,0.86,1


In [18]:
X = df_diff.drop("Result", axis=1)
poly = PolynomialFeatures(degree=3, include_bias=False)
X_poly = poly.fit_transform(X)

pca = PCA(n_components=20)
X_p = pca.fit_transform(X_poly)
model = lgb.LGBMClassifier(
    n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42, verbose=-1
)

cross_val_score(model, X_p, df_diff["Result"], cv=5).mean()

c:\Users\user\anaconda3\envs\ml_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\user\anaconda3\envs\ml_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\user\anaconda3\envs\ml_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\user\anaconda3\envs\ml_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\user\anaconda3\envs\ml_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature nam

np.float64(0.4558441558441559)

In [19]:
X = df_diff.drop("Result", axis=1)
poly = PolynomialFeatures(degree=3, include_bias=False)
X_poly = poly.fit_transform(X)

pca = PCA(n_components=60)
X_p = pca.fit_transform(X_poly)

model = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=1, verbose=-1)
le = LabelEncoder()
y_encoded = le.fit_transform(df_diff["Result"])
cross_val_score(model, X_p, y_encoded, cv=5).mean()

c:\Users\user\anaconda3\envs\ml_env\Lib\site-packages\xgboost\training.py:183: UserWarning: [21:28:28] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "verbose" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\user\anaconda3\envs\ml_env\Lib\site-packages\xgboost\training.py:183: UserWarning: [21:28:28] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "verbose" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\user\anaconda3\envs\ml_env\Lib\site-packages\xgboost\training.py:183: UserWarning: [21:28:28] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "verbose" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\user\anaconda3\envs\ml_env\Lib\site-packages\xgboost\training.py:183: UserWarning: [21:28:28] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "verbose" } are not used.

  bst.update

np.float64(0.48462064251537934)

In [20]:
df_diff[df_diff["Result"] == 0]

,Q_WR,Q_GR,F_Rank,F_Point,F_Rd,F_Pd,Avg_Apps,Avg_Age,Avg_Famous,FS_0,...,FS_5,FS_6,FS_7,FS_8,FS_9,FS_10,FS_11,FS_12,FS_13,Result
21,-0.09,-1.257390,10.00,-165.36,-2.25,100.14,-0.46154,-0.46,-2,-5.81,...,-3.27,-0.74,-2.53,-2.22,-6.02,-3.37,-3.52,-4.09,3.16,0
32,0.01,0.159420,23.34,-83.84,24.00,-85.84,-0.30769,-0.54,0,-4.11,...,-1.77,7.54,0.93,-0.73,-5.19,6.47,-1.71,-3.82,-5.85,0
39,0.07,1.363064,-7.34,45.62,-15.84,265.62,-0.23077,-1.47,3,-0.41,...,0.53,-0.10,0.68,-6.28,-3.57,-5.87,-0.40,2.33,1.60,0
46,0.29,1.105925,-48.00,228.04,-37.92,97.04,0.00000,1.07,5,11.27,...,12.62,2.96,10.66,9.35,14.89,1.77,1.69,11.93,10.03,0
47,0.09,0.768576,-15.33,117.11,-1.08,-159.22,0.53846,1.23,13,8.54,...,11.27,-4.04,4.38,3.46,9.66,-4.96,1.75,11.23,7.14,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
369,-0.01,-0.407927,-22.66,120.33,0.00,0.00,0.00000,0.74,8,2.02,...,-0.15,9.24,-7.07,-6.59,-2.18,-2.28,3.27,3.20,-1.71,0
371,0.04,0.468209,15.33,-45.33,0.00,0.00,-0.08695,-3.65,0,-7.62,...,0.55,0.32,-1.10,-2.38,-1.15,1.41,-9.02,-2.40,1.00,0
377,-0.03,-0.429806,-13.00,59.66,0.00,0.00,0.08696,1.39,-1,3.90,...,-2.52,-1.03,-8.16,-7.36,-6.07,-3.30,-1.94,3.51,4.60,0
378,-0.01,-0.399229,-6.00,32.66,0.00,0.00,0.30435,0.00,8,10.79,...,7.60,6.88,13.28,17.07,11.32,7.02,-2.83,7.12,12.74,0


In [21]:
df_diff_no_draw = df_diff[df_diff["Result"] != 0].copy()

In [22]:
df_diff_no_draw

,Q_WR,Q_GR,F_Rank,F_Point,F_Rd,F_Pd,Avg_Apps,Avg_Age,Avg_Famous,FS_0,...,FS_5,FS_6,FS_7,FS_8,FS_9,FS_10,FS_11,FS_12,FS_13,Result
0,-0.11,1.058700,5.66,-87.00,17.24,-454.42,-0.15385,1.12,2,-1.34,...,5.92,1.93,2.08,-5.54,1.69,-1.96,-0.25,-1.94,-0.90,1
1,0.03,-1.576104,-27.33,149.68,24.00,-345.82,0.23077,1.11,1,0.76,...,5.59,2.55,4.92,5.29,1.41,0.62,-5.60,3.38,7.11,1
2,0.12,-0.225367,-35.66,270.36,17.34,-293.97,0.42308,0.38,5,7.76,...,2.32,4.97,5.03,10.26,2.95,5.73,-2.80,9.47,11.02,1
3,-0.02,2.409437,-2.67,33.68,10.58,-402.57,0.03846,0.39,6,5.66,...,2.65,4.35,2.19,-0.57,3.23,3.15,2.55,4.15,3.01,1
4,-0.16,-0.476359,32.66,-186.23,-28.17,528.27,-0.38461,-0.57,-9,-5.61,...,-5.22,-1.85,-9.21,-3.71,-6.35,-1.07,2.82,-5.97,-6.29,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
376,0.10,0.233593,-52.34,226.33,0.00,0.00,0.47826,4.74,5,11.24,...,12.29,16.15,3.86,4.57,8.95,6.19,0.48,12.60,15.40,1
379,0.19,2.044174,-35.00,171.66,0.00,0.00,0.26087,-1.57,2,15.16,...,11.29,9.27,9.57,11.66,17.95,12.63,6.51,14.62,1.98,1
381,-0.09,-0.556057,14.00,-61.33,0.00,0.00,-0.26087,-2.26,0,-3.81,...,-5.05,-8.38,-8.04,-6.34,-9.42,-6.21,-0.31,-11.34,-2.99,-1
382,-0.02,-0.949593,-37.00,156.00,0.00,0.00,-0.17391,1.83,10,12.19,...,-7.00,6.19,-1.62,1.48,4.52,5.47,10.96,6.95,0.86,1


In [23]:
# 무승부를 포함한 데이터로 뽑았을때
model = XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=1)
le = LabelEncoder()
y_encoded = le.fit_transform(df_diff["Result"])
results = cross_validate(
    model,
    df_diff.drop("Result", axis=1),
    y_encoded,
    cv=5,
    scoring=["precision_macro", "recall_macro", "accuracy"],
)
for key in results:
    mean_val = np.mean(results[key])
    print(f"{key.split('_')[-1]} 평균: {mean_val:.4f}")

c:\Users\user\anaconda3\envs\ml_env\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\user\anaconda3\envs\ml_env\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\user\anaconda3\envs\ml_env\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape

time 평균: 0.0646
time 평균: 0.0091
macro 평균: 0.3968
macro 평균: 0.4804
accuracy 평균: 0.5989


c:\Users\user\anaconda3\envs\ml_env\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [24]:
# 무승부 제외
model = XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=1)
le = LabelEncoder()
y_encoded = le.fit_transform(df_diff_no_draw["Result"])
results = cross_validate(
    model,
    df_diff_no_draw.drop("Result", axis=1),
    y_encoded,
    cv=5,
    scoring=["accuracy", "precision_macro", "recall_macro"],
)
for key in results:
    mean_val = np.mean(results[key])
    print(f"{key} 평균: {mean_val:.4f}")

fit_time 평균: 0.0314
score_time 평균: 0.0086
test_accuracy 평균: 0.7382
test_precision_macro 평균: 0.7376
test_recall_macro 평균: 0.7306


In [25]:
df_diff_no_draw.drop("Result", axis=1).columns

Index(['Q_WR', 'Q_GR', 'F_Rank', 'F_Point', 'F_Rd', 'F_Pd', 'Avg_Apps',
       'Avg_Age', 'Avg_Famous', 'FS_0', 'FS_1', 'FS_2', 'FS_3', 'FS_4', 'FS_5',
       'FS_6', 'FS_7', 'FS_8', 'FS_9', 'FS_10', 'FS_11', 'FS_12', 'FS_13'],
      dtype='object')

In [106]:
from sklearn.model_selection import train_test_split
model = XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=1)
X_train,X_test,y_train,y_test = train_test_split(df_diff_no_draw.drop("Result", axis=1),y_encoded)
model.fit(X_train,y_train)
model.predict_proba(X_test)

array([[0.6410322 , 0.35896775],
       [0.31857848, 0.6814215 ],
       [0.31198627, 0.68801373],
       [0.31198627, 0.68801373],
       [0.38441724, 0.61558276],
       [0.31198627, 0.68801373],
       [0.37721682, 0.6227832 ],
       [0.38441724, 0.61558276],
       [0.6410322 , 0.35896775],
       [0.6410322 , 0.35896775],
       [0.6410322 , 0.35896775],
       [0.6410322 , 0.35896775],
       [0.3932565 , 0.6067435 ],
       [0.31198627, 0.68801373],
       [0.6410322 , 0.35896775],
       [0.31198627, 0.68801373],
       [0.5199797 , 0.48002028],
       [0.31198627, 0.68801373],
       [0.31857848, 0.6814215 ],
       [0.6410322 , 0.35896775],
       [0.6410322 , 0.35896775],
       [0.40056616, 0.59943384],
       [0.31198627, 0.68801373],
       [0.58392215, 0.41607782],
       [0.6410322 , 0.35896775],
       [0.31857848, 0.6814215 ],
       [0.6410322 , 0.35896775],
       [0.6410322 , 0.35896775],
       [0.6410322 , 0.35896775],
       [0.6410322 , 0.35896775],
       [0.